# 01 · Data setup & exploration
Downloads what can be scripted (MedMNIST+), lists manual datasets, builds the vision manifest and sanity-checks class balance and splits.

**Rule:** raw data never goes into git. External test sets (BUS-BRA, Herlev) are kept out of the training manifest.

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
!python data/scripts/download_data.py --list

In [ ]:
!python data/scripts/download_data.py --root data/raw --size 224 --cap 3000

## Manual datasets
Download BUSI, SIPaKMeD and MMOTU per `data/registry.yaml` into `data/raw/<name>/`, plus a small COCO subset into `data/raw/outliers/` for outlier exposure. On Colab, mount Drive and symlink:

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !ln -s /content/drive/MyDrive/rarecare_raw/* data/raw/

In [ ]:
!python data/scripts/build_vision_manifest.py --root data/raw --out data/manifests/vision.csv

In [ ]:
import pandas as pd

m = pd.read_csv("data/manifests/vision.csv")
display(m.groupby(["modality", "split"]).size().unstack(fill_value=0))
display(m[m.label >= 0].groupby(["modality", "label"]).size().unstack(fill_value=0))

In [ ]:
# Leakage check: no file may appear in both splits
dup = set(m[m.split == "train"].path) & set(m[m.split == "val"].path)
assert not dup, f"{len(dup)} files in both splits"
print("no path overlap between train and val")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for ax, (_, r) in zip(axes.flat, m.groupby("modality").head(4).sample(frac=1, random_state=13).head(18).iterrows()):
    ax.imshow(Image.open(r.path).convert("RGB")); ax.set_title(f"{r.modality}\nlabel={r.label}", fontsize=8); ax.axis("off")
plt.tight_layout()